In [1]:
import numpy as np
from scipy.sparse import diags
from scipy.linalg import eigh

In [18]:
# Function to initialize P randomly
def initialize_P(n, k):
    P = np.random.rand(n, k)
    P /= np.sum(P, axis=1)[:, np.newaxis]
    return P

# Function to compute the regularized Laplacian (Lz)
def compute_regularized_laplacian(A, Z):
    return A + Z

# Improved method to estimate number of communities
def estimate_num_communities(A, features):
    degrees = np.sum(A, axis=1)
    D = np.diag(degrees)
    L = D - A  # Laplacian matrix
    eigvals = eigh(L, eigvals_only=True)  # Only get the eigenvalues
    
    # For small graphs, we can count the number of small non-zero eigenvalues
    # (Laplacian matrix tends to have small eigenvalues corresponding to communities)
    eigvals = np.sort(eigvals)
    threshold = 1e-5  # Ignore near-zero eigenvalues
    num_communities = np.sum(eigvals > threshold)
    
    if num_communities == 0:
        num_communities = 2  # Fallback to at least 2 communities if estimate fails
    
    return num_communities

# Symmetric NMF update rule
def update_P(Lz, P, beta=0.5):
    numerator = np.dot(Lz, P)
    denominator = np.dot(np.dot(P, P.T), P)
    denominator[denominator == 0] = 1e-10  # Avoid division by zero
    P = P * ((1 - beta) + beta * numerator / denominator)
    return np.maximum(P, 1e-10)  # Ensure non-negativity

# CDNMF algorithm
def CDNMF(A, features, num_iterations=100, beta=0.5):
    n = A.shape[0]
    
    # Step 1: Estimate the number of communities (k) using spectral analysis
    k = estimate_num_communities(A, features)
    # k = 2  # For simplicity, we will use a fixed number of communities
    print(f"Estimated number of communities: {k}")
    
    # Step 2: Initialize P matrix randomly
    P = initialize_P(n, k)
    
    # Step 3: Initialize regularization matrix Z
    IPR = np.zeros(n)  # Store IPR for each node
    for i in range(n):
        IPR[i] = np.sum(features[i] ** 4)
    
    beta = 0.5  # Regularization parameter
    Z = np.diag(1 + beta * IPR)
    
    # Step 4: Compute regularized Laplacian Lz = A + Z
    Lz = compute_regularized_laplacian(A, Z)
    
    # Step 5: Iteratively update P using the update rule
    for iteration in range(num_iterations):
        P = update_P(Lz, P, beta)
        
        # Normalize P so that each row sums to 1
        P /= np.sum(P, axis=1)[:, np.newaxis]
    
    # Assign nodes to communities based on maximum membership value
    communities = np.argmax(P, axis=1)
    
    return communities, P


In [19]:
# Function to parse .cites and .content files and build the adjacency matrix
def parse_dataset(cites_file, content_file):
    # Parse the .content file
    features = {}
    node_id_to_index = {}
    num_nodes = 0
    with open(content_file, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            node_id = int(parts[0])
            node_id_to_index[node_id] = num_nodes
            feature_vector = np.array([float(x) for x in parts[1:-1]])
            features[num_nodes] = feature_vector
            num_nodes += 1

    # Parse the .cites file
    edges = []
    with open(cites_file, "r") as file:
        for line in file:
            parts = line.strip().split("\t")
            edges.append((int(parts[0]), int(parts[1])))
    
    print(f"Number of nodes: {num_nodes}")
    print(f"Number of edges: {len(edges)}")
    
    # Build the adjacency matrix
    A = np.zeros((num_nodes, num_nodes))
    for edge in edges:
        if edge[0] not in features or edge[1] not in features:
            continue
        x = node_id_to_index[edge[0]]
        y = node_id_to_index[edge[1]]
        A[x, y] = 1
    
    return A, features

In [20]:
# Example usage
if __name__ == "__main__":
    # Load the dataset
    cites_file = "Datasets/cora/cora.cites"
    content_file = "Datasets/cora/cora.content"
    A, features = parse_dataset(cites_file, content_file)

    # Run the CDNMF algorithm
    communities, P = CDNMF(A, features, num_iterations=100, beta=0.5)
    
    # Output community assignment
    print("Community assignment:", communities)
    print("Community membership matrix P:\n", P)

Number of nodes: 2708
Number of edges: 5429
Estimated number of communities: 17
Community assignment: [ 3  6 11 ...  0 11 15]
Community membership matrix P:
 [[2.15347335e-06 4.62134192e-04 1.06238034e-06 ... 3.73600431e-03
  1.03090573e-06 2.97508712e-06]
 [6.20690008e-06 2.94381123e-01 2.13018280e-04 ... 3.26584401e-07
  1.08828290e-06 5.75382120e-05]
 [7.09205205e-07 9.04755872e-07 3.00253016e-08 ... 6.38120262e-09
  2.18123718e-05 4.12960335e-04]
 ...
 [7.47735274e-01 3.36322086e-06 6.19961826e-02 ... 1.28873879e-06
  8.37018425e-07 1.50927007e-05]
 [7.99277759e-06 6.97696298e-08 1.64712307e-04 ... 1.53344815e-06
  3.21667364e-01 1.84692227e-06]
 [1.23601905e-03 4.53877438e-06 2.32597189e-07 ... 1.12260921e-06
  9.88862521e-01 4.78339240e-03]]
